In [ ]:
import pandas as pd
from pathlib import Path
import os

In [ ]:
DATA_DIR = Path.cwd().parent / "data"
DATA_FILE = DATA_DIR / "solicitudes_de_credito.csv.gz"
SUBMISSION_FILE = Path.cwd().parent / "submission" / "solicitudes_de_credito.csv"

In [10]:
datos_credito = pd.read_csv(DATA_FILE, dtype=str, keep_default_na=False, encoding="utf-8-sig", sep=";")
datos_credito

,Unnamed: 0,sexo,tipo_de_emprendimiento,idea_negocio,barrio,estrato,comuna_ciudadano,fecha_de_beneficio,monto_del_credito,línea_credito
0,0,masculino,servicio,peluqueria,santa cruz,1,2.0,28/03/2017,3500000,microempresarial
1,1,femenino,industria,fabrica de,robledo,3,,2018/07/19,7800000,microempresarial
2,2,masculino,comercio,fabrica de,la milagrosa,3,9.0,22/08/2018,7000000,microempresarial
3,3,masculino,industria,fabrica de,manrique central no. 2,2,3.0,15/02/2016,2000000,microempresarial
4,4,femenino,industria,fabrica de,andaluc¿a,02,,6/02/2016,6800000,microempresarial
...,...,...,...,...,...,...,...,...,...,...
11152,11152,femenino,comercio,granero,BARRIO CAICEDO,3,9.0,26/09/2016,6000000,microempresarial
11153,11153,femenino,comercio,comidas rapidas,cabecera san cristobal,2,60.0,22/02/2018,6000000,microempresarial
11154,11154,masculino,industria,fabrica de,campo valdes no. 1,2,4.0,4/05/2018,1800000,MICROEMPRESARIAL
11155,11155,femenino,comercio,comidas rapidas,20 de julio,1,13.0,2017/10/04,1800000,microempresarial


In [ ]:
REQUERIDAS = ["sexo", "tipo_de_emprendimiento", "idea_negocio", "barrio", "estrato",
            "comuna_ciudadano", "fecha_de_beneficio", "monto_del_credito", "línea_credito"]

COLUMNAS_TIPO_TEXTO = ["sexo", "tipo_de_emprendimiento", "idea_negocio",
                       "barrio", "línea_credito"]

COLUMNA_CON_FALTANTES = "comuna_ciudadano"

OBSERVACIONES_NULAS = [" ", "", "nan", "none", "null", "/a", "na", "s/d"]

In [ ]:
# limpieza del texto: minúsculas y separación de palabras con espacio

def limpiar_texto(series):
    series = series.astype("string").str.lower()    # Se pone el contenido de los campos de cada columna en minúsculas, menos los encabezados
    series = series.str.replace("_", " ")           # Se reemplaza el "_" por un espacio en blanco
    series = series.str.replace("\s", " ")          # Se reemplaza cualquier tipo de espacio en blanco por un espacio en blanco
    return series.mask(series.isin(OBSERVACIONES_NULAS))    # La función retorna series con el tratamiento anterior, pero además mask deja explícitamente con NA a todos los campos que contenga los elementos presentes en OBSERVACIONES_NULAS

In [ ]:
def limpiar_estrato(series):
    numero_estrato = series.astype("string").str.extract(r"(\d+)", expand=False)    # Se extrae de cada campo la primera secuencia de digitos que aparezca  hasta que esta se corte por cualquier caracter diferente a un dígito
    return pd.to_numeric(numero_estrato, errors="coerce")                           # Se convierte cada serie de dígitos extraído (extraído como string) a un valor numérico

In [11]:
def limpiar_monto_credito(series):
    series = series.astype("string").str.replace(r"[^\d.,]", "", regex=True)    # Al tener ^ dentro del corchete se indica que todo caracter que no sea un digito coma o punto debe reemplazarse por un string vacío, es decir, eliminarse
    series = series.str.replace(r"[.,]\d{1,2}$", "", regex=True)                # El regex empieza buscando donde esté un . o una , y luego si se tiene una secuencia de 1 o 2 dígitos al final del string se reemplaza por un string vacío, es decir, se elimina
    series = series.str.replace(r"\D", "", regex=True)                          # Se reemplaza cualquier caracter que no sea un dígito por un string vacío, es decir, se elimina
    series = series.mask(series == "")                                          # Todo campo vacío es reemplzada por mask con un NA
    return pd.to_numeric(series, errors="coerce")

In [ ]:
def limpiar_fecha(series):
    series = series.astype("string").str.strip()               # Se eliminanlos espacios en blanco al inicial y al final
    series = series.str.replace(r"[/._]", "-", regex=True)     # Se reemplaza cualquier separador de fechas (/ . _) y se reemplaza por -
    resultado = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")    # pd.NaT significa Not a Time y corresponde a valores faltantes para las fechas. Con index = series.index resultado mantendrpa los mismos índices de series. resultado es uan serie que tiene el índice y dimensiones de series, pero todos sus valroes son NaT

    patrones = [                                               # Lista de tuplas, el primer elemento indica como está compuesta la fecha, el segundo indica las dos formas en que puede interpretarse
        (r"\d{4}-\d{2}-\d{2}", ["%Y-%m-%d", "%Y-%d-%m"]),
        (r"\d{2}-\d{2}-\d{4}", ["%d-%m-%Y", "%m-%d-%Y"])
    ]

    for patron, formato in patrones:
        coincidencias_formato = series.str.fullmatch(patron).fillna(False).astype(bool)   # Para el series tratado anteriormente se dan dos posibles tratamientos: True cuando coincide el patron y False cuando se tiene un dato faltante o cuando no cumple el patron
        for interpretacion in formato:
            por_clasificar = coincidencias_formato & resultado.isna()                     # Se buscan las fechas que cumplen el patron (coincidencia:formato) y que no han sido clasifcadas (resultado isna())
            if not por_clasificar.any():                                                          # Si no hay fechas que cumplan el patron y que no hayan sido clasificadas, se sale del ciclo
                break                                                                     # Se detiene el loop cuando ya no haya en resultados un NA que cumpla el patron (que sea true)
            resultado[por_clasificar] = pd.to_datetime(series[por_clasificar], format=interpretacion, errors="coerce")  # Para todos los elementos en por_clasificar que sean true se hace la conversión a fecha según el formato de interpretacion

    return resultado

In [ ]:
# Lectura de los datos
datos_credito = pd.read_csv(DATA_FILE, dtype=str, sep=";")

datos_credito_final = datos_credito[REQUERIDAS].copy()

In [ ]:
# Validación de columnas de tipo texto
for col in COLUMNAS_TIPO_TEXTO:
    datos_credito_final[col] = limpiar_texto(datos_credito_final[col])

datos_credito_final["estrato"] = limpiar_estrato(datos_credito_final["estrato"])
datos_credito_final["monto_del_credito"] = limpiar_monto_credito(datos_credito_final["monto_del_credito"])
datos_credito_final["fecha_de_beneficio"] = limpiar_fecha(datos_credito_final["fecha_de_beneficio"])

In [ ]:
# Análisis de la columna comuna_ciudadano
datos_credito_final[COLUMNA_CON_FALTANTES] = pd.to_numeric(
    datos_credito_final[COLUMNA_CON_FALTANTES], errors="coerce"
).astype("Int64")

In [ ]:
# Depuración de las filas con datos faltantes presentes en columnas requeridas diferentes a comuna_ciudadano
columnas_requeridas = [c for c in REQUERIDAS if c != COLUMNA_CON_FALTANTES]
datos_credito_final = datos_credito_final.dropna(subset=columnas_requeridas)

In [ ]:
# Ajuste de formato a enteros o fecha
datos_credito_final["estrato"] = datos_credito_final["estrato"].astype("int64")
datos_credito_final["monto_del_credito"] = datos_credito_final["monto_del_credito"].astype("int64")
datos_credito_final["fecha_de_beneficio"] = datos_credito_final["fecha_de_beneficio"].dt.strftime("%Y-%m-%d")

In [ ]:
# Validación de duplicados
datos_credito_final = datos_credito_final.drop_duplicates().reset_index(drop=True)

In [ ]:
# Creación archivo de salida
datos_credito_final.to_csv(SUBMISSION_FILE, sep=";", index=False)

# DIAGNOSTICO FINAL

In [25]:
import unicodedata
import pandas as pd

OUTPUT_FILE = r"C:\Users\SANDRA\Documents\GitHub\std-andres-velez-velez\LAB_06_limpieza_de_solicitudes_de_credito\submission\solicitudes_de_credito.csv"
df = pd.read_csv(OUTPUT_FILE, sep=";")

print(len(df))
print(df.columns.tolist())

COLUMNAS_TIPO_TEXTO = ["sexo", "tipo_de_emprendimiento", "idea_negocio",
                       "barrio", "línea_credito"]

10402
['sexo', 'tipo_de_emprendimiento', 'idea_negocio', 'barrio', 'estrato', 'comuna_ciudadano', 'fecha_de_beneficio', 'monto_del_credito', 'línea_credito']


In [26]:
def quitar_tildes(t):
    if pd.isna(t):
        return t    # Regresa valores faltantes sin cambios
    return "".join(c for c in unicodedata.normalize("NFKD", t) if not unicodedata.combining(c))  # Separa letra y acento, descarta los acentos y vuelve a unir el texto

def clave(serie):
    s = serie.astype("string").map(quitar_tildes).astype("string").str.lower()   # Quita tildes y transforma a minúscula
    return s.str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()    # Quita los espacios a principio y final, reemplaza signos por un espacio

claves = df.copy()
for col in COLUMNAS_TIPO_TEXTO:
    claves[col] = clave(df[col])    # Se aplica la limpieza definida en clave sobre cada columna de tipo texto

solo_tildes = df.copy()
for col in COLUMNAS_TIPO_TEXTO:
    solo_tildes[col] = df[col].astype("string").map(quitar_tildes).astype("string") # Reemplaza cada una por su versión sin tildes

print("Duplicados exactos          :", df.duplicated().sum())
print("Ocultos solo por tildes     :", solo_tildes.duplicated().sum())
print("Ocultos por tildes y signos :", claves.duplicated().sum())
print("Filas que quedarían         :", len(df) - claves.duplicated().sum())

Duplicados exactos          : 0
Ocultos solo por tildes     : 0
Ocultos por tildes y signos : 0
Filas que quedarían         : 10402


In [ ]:
from collections import Counter

# Validar los signos presentes en las columnas de texto
simbolos = Counter()
for col in COLUMNAS_TIPO_TEXTO:
    simbolos.update(c for texto in df[col].dropna() for c in texto
                    if not (c.isalnum() or c == " "))
print(simbolos)

# Validar los pares de filas que son iguales salvo por esos signos
ocultos = claves.duplicated(keep=False)
grupos = claves[ocultos].groupby(list(claves.columns), dropna=False).ngroup()
vista = df.loc[ocultos].assign(grupo=grupos).sort_values("grupo")
print(vista[["grupo"] + COLUMNAS_TIPO_TEXTO].to_string(index=False))

Counter({'.': 1869, '¿': 14})
Empty DataFrame
Columns: [grupo, sexo, tipo_de_emprendimiento, idea_negocio, barrio, línea_credito]
Index: []


Se identifican simbolos adicionales por incluir en la limpieza: 
- No sólo se trata de espacios " ", también está presente otros como: [-]